# Financial RAG Pipeline — Interactive Demo

This notebook demonstrates the end-to-end RAG pipeline for financial document intelligence.

**Components demonstrated:**
- Document ingestion and chunking
- Embedding generation
- Semantic retrieval from ChromaDB
- LLM generation with context
- Grounding evaluation

In [ ]:
import asyncio
import sys
sys.path.insert(0, '..')

from app.config.settings import settings
from app.rag.chunking import FinancialChunker
from app.rag.pipeline import RAGPipeline
from app.models.documents import Document, DocumentMetadata

print(f'Financial RAG Research Assistant v{settings.app_version}')
print(f'LLM Provider: {settings.llm_provider}')
print(f'Embedding Model: {settings.embedding_model}')

## 1. Document Chunking Demo

In [ ]:
# Sample SEC filing text
sample_text = """
ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS

Fiscal Year 2023 Overview
Apple Inc. reported total net sales of $383.3 billion for fiscal year 2023,
representing a decline of 3% compared to $394.3 billion in fiscal year 2022.
Net income was $97.0 billion, or $6.13 diluted earnings per share.

ITEM 1A. RISK FACTORS

Global and regional economic conditions could materially adversely affect the
Company's business, results of operations, financial condition, and growth.
The Company's operations and financial results are subject to various risks.
"""

# Create chunker
chunker = FinancialChunker(chunk_size=128, chunk_overlap=16, strategy='semantic')

# Create document
doc = Document(
    doc_id='aapl_10k_2023',
    title='Apple Inc. 10-K FY2023',
    content=sample_text,
    metadata=DocumentMetadata(
        source='data/raw/sec_filings/AAPL_10K_2023.txt',
        category='sec_filing',
        ticker='AAPL',
        filing_type='10-K',
        fiscal_year='2023',
    )
)

# Chunk the document
chunks = chunker.chunk_document(doc)
print(f'Document chunked into {len(chunks)} chunks')
for i, chunk in enumerate(chunks):
    print(f'  Chunk {i}: {chunk.token_count} tokens — "{chunk.content[:80]}..."')

## 2. Financial Utility Functions

In [ ]:
from app.utils.text_processing import extract_financial_figures, extract_ticker_symbols
from app.utils.financial_utils import format_currency, compute_returns, compute_max_drawdown

# Extract financial figures from text
text = 'Apple reported $383.3 billion in revenue and net income of $97 million. EPS was $6.13.'
figures = extract_financial_figures(text)
print('Extracted financial figures:')
for fig in figures:
    print(f'  {fig["raw"]} -> {format_currency(fig["value"])} ({fig["type"]})')

# Extract ticker symbols
text2 = 'AAPL, MSFT, and GOOGL are mega-cap technology stocks. $NVDA is a semiconductor leader.'
tickers = extract_ticker_symbols(text2)
print(f'\nExtracted tickers: {tickers}')

# Compute portfolio metrics
prices = [100.0, 115.0, 108.0, 125.0, 118.0, 140.0]
returns = compute_returns(prices)
max_dd = compute_max_drawdown(prices)
print(f'\nPrice series: {prices}')
print(f'Period returns: {[round(r*100, 2) for r in returns]}%')
print(f'Max drawdown: {max_dd*100:.2f}%')

## 3. Governance Checks Demo

In [ ]:
import asyncio
from app.governance.guardrails import GovernanceService

gov = GovernanceService()

# Test with clean financial content
clean_content = 'Apple Inc. reported $383 billion in revenue for FY2023 with 45.2% gross margin.'
result = asyncio.run(gov.run_checks(clean_content, check_types=['pii', 'injection', 'compliance']))
print(f'Clean content — Passed: {result.overall_passed}')
print(f'  PII detected: {result.pii_detected}')
print(f'  Injection detected: {result.injection_detected}')

# Test with PII
pii_content = 'Contact John at john.doe@example.com, SSN: 123-45-6789'
result2 = asyncio.run(gov.run_checks(pii_content, check_types=['pii']))
print(f'\nPII content — Passed: {result2.overall_passed}')
print(f'  PII detected: {result2.pii_detected}')

# Redact PII
redacted = gov.redact_pii(pii_content)
print(f'  Redacted: {redacted}')

## 4. Evaluation Pipeline Demo

In [ ]:
import asyncio
from app.evaluation.grounding import GroundingEvaluator
from app.evaluation.hallucination import HallucinationDetector
from app.evaluation.scoring import QualityScorer

# Setup evaluators
grounding_eval = GroundingEvaluator(threshold=0.85)
hallucination_detector = HallucinationDetector()
quality_scorer = QualityScorer()

# Sample data
query = 'What was Apple revenue in FY2023?'
context = ['Apple Inc. reported total net sales of $383.3 billion for fiscal year 2023, down 3% from $394.3 billion in FY2022.']
response = 'Apple reported $383.3 billion in revenue for fiscal year 2023, representing a 3% decline from the prior year.'

# Grounding evaluation
grounding = asyncio.run(grounding_eval.score_grounding(query, response, context))
print(f'Grounding Score: {grounding["grounding_score"]:.3f}')
print(f'Hallucination Risk: {grounding["hallucination_risk"]}')
print(f'Passed: {grounding["passed"]}')

# Hallucination detection
hall = asyncio.run(hallucination_detector.detect(response, context, query))
print(f'\nHallucination Detected: {hall["hallucination_detected"]}')
print(f'Risk Level: {hall["risk_level"]}')

# Quality scoring
quality = asyncio.run(quality_scorer.score(query, response, context))
print(f'\nOverall Quality: {quality["overall_quality"]:.3f}')
print(f'Dimension Scores: {quality["dimension_scores"]}')